# Interactive PaCMAP 2 (Configurable Mix + Multimodal Compatibility)

This notebook builds an interactive MAEST PaCMAP for a selected mix CSV (for example, `music/aries-mix/aries_mix_tracks.csv` or `music/ara-mix/ara_mix_tracks.csv`) and uses multimodal compatibility (MAEST+Chroma+Tempo) for click-based top-25 recommendations.

Set `MIX_MODE` in the first code cell to `aries-mix`, `ara-mix`, or `both`; file paths and export names are derived automatically.


In [1]:
# Interactive PaCMAP for a configurable mix:
# - 2D coordinates from MAEST embeddings only
# - similarity/recommendations from multimodal compatibility (MAEST + Chroma + Tempo)
# - metadata from CSV tags (title, artists, key, bpm, genre)
# - includes estimated tempo_bpm / tempo_confidence from tempo embeddings

from __future__ import annotations

import csv
import importlib
import inspect
import json
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import webbrowser


def detect_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / 'data').exists() and (c / 'notebooks').exists() and (c / 'music').exists():
            return c
    return cwd


PROJECT_ROOT = detect_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import djprojectexploration.audio_snippets as _audio_snippets_mod
import djprojectexploration.multimodal_compatibility as _multimodal_compat_mod

importlib.reload(_audio_snippets_mod)
importlib.reload(_multimodal_compat_mod)

ensure_cached_snippet = _audio_snippets_mod.ensure_cached_snippet
compatible_song_distribution = _multimodal_compat_mod.compatible_song_distribution
load_feature_set = _multimodal_compat_mod.load_aries_mix_feature_set
SongFeatureSet = _multimodal_compat_mod.SongFeatureSet
SongMetadata = _multimodal_compat_mod.SongMetadata

if 'harmonic_exact_weight' not in inspect.signature(compatible_song_distribution).parameters:
    raise RuntimeError(
        'Loaded multimodal_compatibility does not expose harmonic params. '
        'Restart kernel and rerun this cell.'
    )

try:
    import pacmap
except ImportError as exc:
    raise ImportError('PaCMAP is not installed. Install with: pip install pacmap') from exc
# Mix selection
MIX_MODE = 'both'  # one of: 'aries-mix', 'ara-mix', 'both'
if MIX_MODE == 'both':
    MIX_SLUGS = ['aries-mix', 'ara-mix']
else:
    MIX_SLUGS = [str(MIX_MODE).strip()]

MIX_CONFIGS: list[dict[str, Path | str]] = []
for mix_slug in MIX_SLUGS:
    tracklist_csv = PROJECT_ROOT / 'music' / mix_slug / f"{mix_slug.replace('-', '_')}_tracks.csv"
    MIX_CONFIGS.append({
        'mix_slug': mix_slug,
        'tracklist_csv': tracklist_csv,
        'music_dir': tracklist_csv.parent,
        'snippet_cache_dir': PROJECT_ROOT / 'data' / 'snippets' / tracklist_csv.stem,
    })

if not MIX_CONFIGS:
    raise ValueError('MIX_CONFIGS is empty. Select at least one mix.')

MAEST_DIR = PROJECT_ROOT / 'data' / 'maest_embeddings'
CHROMA_DIR = PROJECT_ROOT / 'data' / 'chroma_embeddings'
TEMPO_DIR = PROJECT_ROOT / 'data' / 'tempo_embeddings'

# Multimodal compatibility params
MULTIMODAL_MAEST_WEIGHT = 0.60
MULTIMODAL_CHROMA_WEIGHT = 0.25
MULTIMODAL_TEMPO_WEIGHT = 0.15
MULTIMODAL_TEMPERATURE = 0.08
MULTIMODAL_CANDIDATE_TOP_K = None

# Tempo similarity params
TEMPO_BANDWIDTH = 0.06
TEMPO_DECAY = 0.5
TEMPO_ALLOW_OCTAVE = True
TEMPO_OCTAVE_PENALTY = 0.5
TEMPO_SIMILARITY_SHAPE = 'gaussian'  # one of: gaussian, exp, flat+exp, softflat+exp
TEMPO_SOFTFLAT_SHARPNESS = 8.0
TEMPO_USE_CONFIDENCE = False

# Harmonic fifth-aware kernel params (for chroma similarity)
HARMONIC_EXACT_WEIGHT = 1.0
HARMONIC_FIRST_FIFTH_WEIGHT = 0.0
HARMONIC_SECOND_FIFTH_WEIGHT = 0.0
HARMONIC_OTHER_WEIGHT = 0.0
HARMONIC_SELF_NORMALIZE = True

# PaCMAP params (MAEST-only)
PACMAP_RANDOM_STATE = 7777
PACMAP_MN_RATIO = 0.5
PACMAP_FP_RATIO = 1.5
PACMAP_MAX_NEIGHBORS = 10

# Snippet cache params
SNIPPET_SECONDS = 8.0
SNIPPET_MIDDLE_FRACTION = 0.66
SNIPPET_HOP_SECONDS = 0.25
SNIPPET_CACHE_OVERWRITE = False

# HTML export
HTML_EXPORT_DIR = PROJECT_ROOT / 'data' / 'exports'
if len(MIX_CONFIGS) == 1:
    dataset_tag = str(MIX_CONFIGS[0]['tracklist_csv'].stem)
else:
    dataset_tag = '__'.join(str(cfg['tracklist_csv'].stem) for cfg in MIX_CONFIGS)
INTERACTIVE_HTML_PATH = HTML_EXPORT_DIR / f'{dataset_tag}_interactive_pacmap_multimodal.html'
PLOT_DIV_ID = f'{dataset_tag}_pacmap_multimodal_plot'.replace('-', '_')
MIX_TITLE = ' + '.join(str(cfg['mix_slug']) for cfg in MIX_CONFIGS)

for cfg in MIX_CONFIGS:
    tracklist_csv = Path(cfg['tracklist_csv'])
    if not tracklist_csv.exists():
        raise FileNotFoundError(f'Tracklist CSV not found: {tracklist_csv}')
if not MAEST_DIR.exists():
    raise FileNotFoundError(f'MAEST dir not found: {MAEST_DIR}')
if not CHROMA_DIR.exists():
    raise FileNotFoundError(f'Chroma dir not found: {CHROMA_DIR}')
if not TEMPO_DIR.exists():
    raise FileNotFoundError(f'Tempo dir not found: {TEMPO_DIR}')


def norm_token(value: str) -> str:
    return Path(str(value).strip()).name.lower()


def row_token(row: dict) -> str | None:
    for key in ('mp3_name', 'filename', 'filepath', 'location'):
        val = (row.get(key) or '').strip()
        if val:
            return norm_token(val)
    return None


def resolve_audio_path(
    row: dict,
    fallback_filename: str,
    *,
    tracklist_csv: Path,
    music_dir: Path,
) -> Path | None:
    for key in ('filepath', 'location'):
        val = (row.get(key) or '').strip()
        if val:
            p = Path(val).expanduser()
            if p.exists():
                return p.resolve()

    mp3_name = (row.get('mp3_name') or row.get('filename') or fallback_filename or '').strip()
    if not mp3_name:
        return None

    rel = Path(mp3_name)
    candidates = [
        tracklist_csv.parent / rel,
        PROJECT_ROOT / rel,
        music_dir / rel.name,
        PROJECT_ROOT / 'music' / rel.name,
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return None


# Load aligned multimodal feature set(s) and merge with unique global track numbers.
combined_metadata: list[SongMetadata] = []
combined_maest: list[np.ndarray] = []
combined_chroma: list[np.ndarray] = []
combined_chroma_pitch: list[np.ndarray] = []
combined_tempo_bpm: list[float] = []
combined_tempo_conf: list[float] = []
records: list[dict] = []

for cfg in MIX_CONFIGS:
    mix_slug = str(cfg['mix_slug'])
    tracklist_csv = Path(cfg['tracklist_csv'])
    music_dir = Path(cfg['music_dir'])
    snippet_cache_dir = Path(cfg['snippet_cache_dir'])

    mix_features = load_feature_set(
        mix_csv_path=tracklist_csv,
        maest_dir=MAEST_DIR,
        chroma_dir=CHROMA_DIR,
        tempo_dir=TEMPO_DIR,
        chroma_use_base_only=True,
    )

    with tracklist_csv.open('r', encoding='utf-8', newline='') as f:
        csv_rows = list(csv.DictReader(f))

    csv_by_token: dict[str, dict] = {}
    for row in csv_rows:
        token = row_token(row)
        if token and token not in csv_by_token:
            csv_by_token[token] = row

    for local_i, meta in enumerate(mix_features.metadata):
        global_idx = len(records)
        global_track_number = global_idx + 1

        token = norm_token(meta.filename)
        row = csv_by_token.get(token, {})

        title = (row.get('title') or meta.title or Path(meta.filename).stem).strip()
        artists = (row.get('artists') or meta.artist or '').strip()
        key_tag = (row.get('key') or '').strip()
        bpm_tag = (row.get('bpm') or '').strip()
        genre = (row.get('genre') or meta.genre or 'Unknown').strip() or 'Unknown'
        track_num_tag = (row.get('track_number') or row.get('#') or str(meta.track_number)).strip()

        est_bpm = float(mix_features.tempo_bpm[local_i])
        est_conf = float(mix_features.tempo_confidence[local_i])

        audio_path = resolve_audio_path(
            row,
            fallback_filename=meta.filename,
            tracklist_csv=tracklist_csv,
            music_dir=music_dir,
        )
        try:
            snippet = ensure_cached_snippet(
                audio_path=audio_path,
                output_dir=snippet_cache_dir,
                key=meta.filename,
                snippet_seconds=SNIPPET_SECONDS,
                middle_fraction=SNIPPET_MIDDLE_FRACTION,
                hop_seconds=SNIPPET_HOP_SECONDS,
                overwrite=SNIPPET_CACHE_OVERWRITE,
                project_root=PROJECT_ROOT,
            )
        except Exception:
            snippet = None

        if snippet is None:
            snippet_uri, start_s, end_s, snippet_rms = '', 0.0, 0.0, 0.0
            snippet_path = ''
        else:
            snippet_file = snippet.snippet_path.resolve()
            start_s = float(snippet.start_seconds)
            end_s = float(snippet.end_seconds)
            snippet_rms = float(snippet.rms)
            snippet_path = str(snippet_file)
            try:
                snippet_uri = snippet_file.relative_to(INTERACTIVE_HTML_PATH.parent.resolve()).as_posix()
            except ValueError:
                snippet_uri = snippet_file.as_uri()

        combined_metadata.append(SongMetadata(
            track_number=int(global_track_number),
            title=str(meta.title),
            artist=str(meta.artist),
            filename=str(meta.filename),
            genre=str(meta.genre),
        ))
        combined_maest.append(np.asarray(mix_features.maest[local_i], dtype=np.float32))
        combined_chroma.append(np.asarray(mix_features.chroma[local_i], dtype=np.float32))
        combined_chroma_pitch.append(np.asarray(mix_features.chroma_pitch[local_i], dtype=np.float32))
        combined_tempo_bpm.append(est_bpm)
        combined_tempo_conf.append(est_conf)

        records.append({
            'idx': global_idx,
            'global_track_number': int(global_track_number),
            'track_number': str(track_num_tag),
            'filename': meta.filename,
            'title': title,
            'artists': artists,
            'genre': genre,
            'key': key_tag,
            'csv_bpm': bpm_tag,
            'est_bpm': est_bpm,
            'est_conf': est_conf,
            'audio_path': '' if audio_path is None else str(audio_path),
            'snippet_uri': snippet_uri,
            'snippet_start': start_s,
            'snippet_end': end_s,
            'snippet_rms': snippet_rms,
            'snippet_path': snippet_path,
            'mix_slug': mix_slug,
        })

if len(records) < 2:
    raise ValueError('Need at least 2 aligned tracks with embeddings.')

features = SongFeatureSet(
    metadata=combined_metadata,
    maest=np.vstack(combined_maest).astype(np.float32),
    chroma=np.vstack(combined_chroma).astype(np.float32),
    chroma_pitch=np.vstack(combined_chroma_pitch).astype(np.float32),
    tempo_bpm=np.asarray(combined_tempo_bpm, dtype=np.float32),
    tempo_confidence=np.asarray(combined_tempo_conf, dtype=np.float32),
)

global_track_number_to_idx = {
    int(meta.track_number): i
    for i, meta in enumerate(features.metadata)
}

# Build per-source multimodal ranking payload.
similarity_payload: dict[str, dict[str, object]] = {}

for src_idx, meta in enumerate(features.metadata):
    dist_rows = compatible_song_distribution(
        features=features,
        seed_track_number=int(meta.track_number),
        maest_weight=MULTIMODAL_MAEST_WEIGHT,
        chroma_weight=MULTIMODAL_CHROMA_WEIGHT,
        harmonic_exact_weight=HARMONIC_EXACT_WEIGHT,
        harmonic_first_fifth_weight=HARMONIC_FIRST_FIFTH_WEIGHT,
        harmonic_second_fifth_weight=HARMONIC_SECOND_FIFTH_WEIGHT,
        harmonic_other_weight=HARMONIC_OTHER_WEIGHT,
        harmonic_self_normalize=HARMONIC_SELF_NORMALIZE,
        tempo_weight=MULTIMODAL_TEMPO_WEIGHT,
        temperature=MULTIMODAL_TEMPERATURE,
        candidate_top_k=MULTIMODAL_CANDIDATE_TOP_K,
        tempo_bandwidth=TEMPO_BANDWIDTH,
        tempo_decay=TEMPO_DECAY,
        tempo_allow_octave=TEMPO_ALLOW_OCTAVE,
        tempo_octave_penalty=TEMPO_OCTAVE_PENALTY,
        tempo_similarity_shape=TEMPO_SIMILARITY_SHAPE,
        tempo_softflat_sharpness=TEMPO_SOFTFLAT_SHARPNESS,
        tempo_use_confidence=TEMPO_USE_CONFIDENCE,
    )

    probs = np.asarray([float(r['probability']) for r in dist_rows], dtype=np.float64)
    nz_probs = probs[probs > 0.0]
    sorted_probs = np.sort(nz_probs)[::-1] if nz_probs.size else np.array([], dtype=np.float64)
    entropy_nats = float(-np.sum(nz_probs * np.log(np.clip(nz_probs, 1e-12, 1.0)))) if nz_probs.size else 0.0

    top_rows = []
    for rank, row in enumerate(dist_rows[:25], start=1):
        cand_global_track_number = int(row.get('track_number', -1))
        cand_idx = global_track_number_to_idx.get(cand_global_track_number)
        cand_record = records[cand_idx] if cand_idx is not None else None

        if cand_record is None:
            cand_title = (row.get('title') or Path(str(row.get('filename', ''))).stem).strip()
            cand_artists = (row.get('artist') or '').strip()
            cand_key = ''
            cand_csv_bpm = ''
            cand_genre = (row.get('genre') or 'Unknown').strip() or 'Unknown'
            cand_est_bpm = 0.0
            cand_est_conf = 0.0
            cand_mix_slug = ''
            cand_track_number = ''
        else:
            cand_title = str(cand_record['title'])
            cand_artists = str(cand_record['artists'])
            cand_key = str(cand_record['key'])
            cand_csv_bpm = str(cand_record['csv_bpm'])
            cand_genre = str(cand_record['genre'])
            cand_est_bpm = float(cand_record['est_bpm'])
            cand_est_conf = float(cand_record['est_conf'])
            cand_mix_slug = str(cand_record['mix_slug'])
            cand_track_number = str(cand_record['track_number'])

        top_rows.append({
            'rank': rank,
            'idx': -1 if cand_idx is None else int(cand_idx),
            'global_track_number': cand_global_track_number,
            'track_number': cand_track_number,
            'mix_slug': cand_mix_slug,
            'title': cand_title,
            'artists': cand_artists,
            'genre': cand_genre,
            'key': cand_key,
            'csv_bpm': cand_csv_bpm,
            'est_bpm': cand_est_bpm,
            'est_conf': cand_est_conf,
            'filename': str(row.get('filename', '')),
            'compatibility_score': float(row.get('compatibility_logit', 0.0)),
            'maest_similarity': float(row.get('maest_similarity', 0.0)),
            'chroma_similarity': float(row.get('chroma_similarity', 0.0)),
            'tempo_similarity': float(row.get('tempo_similarity', 0.0)),
            'probability': float(row.get('probability', 0.0)),
        })

    mass_top_25 = float(np.sum(sorted_probs[:25])) if sorted_probs.size else 0.0
    similarity_payload[str(src_idx)] = {
        'top_25': top_rows,
        'distribution': {
            'nonzero_count': int(nz_probs.size),
            'max_probability': float(sorted_probs[0]) if sorted_probs.size else 0.0,
            'mean_probability': float(np.mean(nz_probs)) if nz_probs.size else 0.0,
            'median_probability': float(np.median(nz_probs)) if nz_probs.size else 0.0,
            'entropy_nats': entropy_nats,
            'mass_top_1': float(np.sum(sorted_probs[:1])) if sorted_probs.size else 0.0,
            'mass_top_5': float(np.sum(sorted_probs[:5])) if sorted_probs.size else 0.0,
            'mass_top_10': float(np.sum(sorted_probs[:10])) if sorted_probs.size else 0.0,
            'mass_top_25': mass_top_25,
            'mass_remaining': float(max(0.0, 1.0 - mass_top_25)),
        },
    }

sim_json = json.dumps(similarity_payload, ensure_ascii=False)

# MAEST-only PaCMAP layout.
maest = np.asarray(features.maest, dtype=np.float32)
reducer = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=min(PACMAP_MAX_NEIGHBORS, max(2, maest.shape[0] - 1)),
    MN_ratio=PACMAP_MN_RATIO,
    FP_ratio=PACMAP_FP_RATIO,
    random_state=PACMAP_RANDOM_STATE,
)
coords = reducer.fit_transform(maest)

genres = sorted({r['genre'] for r in records}, key=lambda g: g.lower())
palette = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b',
    '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#393b79', '#637939',
]
genre_color = {g: palette[i % len(palette)] for i, g in enumerate(genres)}

fig = go.Figure()
for genre in genres:
    pts = [r for r in records if r['genre'] == genre]
    idxs = [p['idx'] for p in pts]
    custom = [[
        p['title'], p['artists'], p['genre'], p['key'], p['csv_bpm'],
        p['est_bpm'], p['est_conf'], p['track_number'], p['filename'],
        p['snippet_uri'], p['snippet_start'], p['snippet_end'], p['snippet_rms'], p['idx'], p['mix_slug'],
    ] for p in pts]

    fig.add_scatter(
        x=coords[idxs, 0],
        y=coords[idxs, 1],
        mode='markers',
        name=genre,
        marker=dict(size=9, color=genre_color[genre], line=dict(color='black', width=0.5)),
        customdata=custom,
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Artists: %{customdata[1]}<br>'
            'Genre: %{customdata[2]}<br>'
            'Mix: %{customdata[14]}<br>'
            'Key: %{customdata[3]}<br>'
            'CSV BPM: %{customdata[4]}<br>'
            'Estimated BPM: %{customdata[5]:.2f} (conf=%{customdata[6]:.3f})<br>'
            'Track #: %{customdata[7]}<br>'
            'File: %{customdata[8]}<extra></extra>'
        ),
    )

xmin, xmax = float(np.min(coords[:, 0])), float(np.max(coords[:, 0]))
ymin, ymax = float(np.min(coords[:, 1])), float(np.max(coords[:, 1]))
xmid, ymid = (xmin + xmax) / 2.0, (ymin + ymax) / 2.0
xspan0, yspan0 = max(1.0, (xmax - xmin) * 0.55), max(1.0, (ymax - ymin) * 0.55)

zoom_factors = [0.4, 0.7, 1.0, 1.5, 2.2]
zoom_buttons = []
for factor in zoom_factors:
    zoom_buttons.append(dict(
        label=f'Zoom {factor:.1f}x',
        method='relayout',
        args=[{
            'xaxis.range': [xmid - xspan0 * factor, xmid + xspan0 * factor],
            'yaxis.range': [ymid - yspan0 * factor, ymid + yspan0 * factor],
        }],
    ))
zoom_buttons.append(dict(label='Reset', method='relayout', args=[{'xaxis.autorange': True, 'yaxis.autorange': True}]))

fig.update_layout(
    title=f'{MIX_TITLE}: Interactive MAEST PaCMAP + Multimodal Compatibility (click for audio + top-25)',
    xaxis_title='PaCMAP-1 (MAEST)',
    yaxis_title='PaCMAP-2 (MAEST)',
    width=1180,
    height=780,
    margin=dict(t=130),
    legend_title_text='Genre (from CSV)',
    updatemenus=[dict(
        type='buttons',
        direction='right',
        showactive=False,
        x=0.0,
        xanchor='left',
        y=1.13,
        yanchor='top',
        buttons=zoom_buttons,
    )],
)

plot_div_id = PLOT_DIV_ID
plot_html = fig.to_html(include_plotlyjs=True, full_html=False, div_id=plot_div_id)

meta_audio_html = '''
<div style="margin-top:12px;">
  <div id="track-meta" style="padding:8px;border:1px solid #ddd;border-radius:6px;font-size:13px;">
    Click a point to auto-play its high-RMS snippet and show top-25 multimodal compatible songs.
  </div>
  <audio id="track-audio" controls style="width:100%;margin-top:8px;"></audio>
  <div id="similarity-panel" style="margin-top:10px;padding:8px;border:1px solid #ddd;border-radius:6px;font-size:12px;">
    Similarity panel will appear here after you click a point.
  </div>
</div>
'''

weights_json = json.dumps({
    'maest': float(MULTIMODAL_MAEST_WEIGHT),
    'chroma': float(MULTIMODAL_CHROMA_WEIGHT),
    'tempo': float(MULTIMODAL_TEMPO_WEIGHT),
}, ensure_ascii=False)
idx_to_point_json = json.dumps({
    str(i): [float(coords[i, 0]), float(coords[i, 1])]
    for i in range(coords.shape[0])
}, ensure_ascii=False)

script = '''
<script>
(function() {
  const simMap = __SIM_MAP__;
  const modelWeights = __MODEL_WEIGHTS__;
  const idxToPoint = __IDX_TO_POINT__;
  const plot = document.getElementById('__PLOT_ID__');
  const meta = document.getElementById('track-meta');
  const audio = document.getElementById('track-audio');
  const panel = document.getElementById('similarity-panel');
  if (!plot || !meta || !audio || !panel) return;

  let dynamicLinkTraceIndices = [];

  function fmtPct(v) {
    const n = Number(v || 0);
    return (n * 100).toFixed(2) + '%';
  }

  function fmtNum(v, digits) {
    return Number(v || 0).toFixed(digits);
  }

  function clearDynamicLinks() {
    if (!dynamicLinkTraceIndices.length) return;
    if (!window.Plotly || !Plotly.deleteTraces) {
      dynamicLinkTraceIndices = [];
      return;
    }
    const toDelete = dynamicLinkTraceIndices.slice().sort((a, b) => b - a);
    try {
      Plotly.deleteTraces(plot, toDelete);
    } catch (err) {
      /* no-op */
    }
    dynamicLinkTraceIndices = [];
  }

  function addTop5Links(sourceIdx, matches) {
    clearDynamicLinks();
    if (!window.Plotly || !Plotly.addTraces) return;

    const src = idxToPoint[String(sourceIdx ?? '')];
    if (!src) return;

    const linkRows = (matches || []).slice(0, 5);
    const linkTraces = [];
    for (const m of linkRows) {
      const dst = idxToPoint[String(m.idx ?? '')];
      if (!dst) continue;
      const prob = Number(m.probability || 0);
      const alpha = Math.max(0.25, Math.min(0.95, 0.25 + prob * 3.0));
      const width = Math.max(1.2, Math.min(4.0, 1.2 + prob * 10.0));
      linkTraces.push({
        type: 'scatter',
        mode: 'lines',
        x: [Number(src[0]), Number(dst[0])],
        y: [Number(src[1]), Number(dst[1])],
        line: {
          color: 'rgba(30,30,30,' + alpha.toFixed(3) + ')',
          width: width,
          dash: 'dot',
        },
        hoverinfo: 'skip',
        showlegend: false,
      });
    }

    if (!linkTraces.length) return;
    const baseLen = (plot.data || []).length;
    const newIndices = Array.from({ length: linkTraces.length }, (_, i) => baseLen + i);
    try {
      const addResult = Plotly.addTraces(plot, linkTraces);
      if (addResult && typeof addResult.then === 'function') {
        addResult.then(() => {
          dynamicLinkTraceIndices = newIndices;
        }).catch(() => {
          dynamicLinkTraceIndices = [];
        });
      } else {
        dynamicLinkTraceIndices = newIndices;
      }
    } catch (err) {
      dynamicLinkTraceIndices = [];
    }
  }

  plot.on('plotly_click', function(ev) {
    if (!ev || !ev.points || !ev.points.length) return;
    const p = ev.points[0];
    const c = p.customdata || [];

    const title = c[0] || 'Unknown title';
    const artists = c[1] || '';
    const genre = c[2] || '';
    const key = c[3] || '';
    const csvBpm = c[4] || '';
    const estBpm = Number(c[5] || 0);
    const estConf = Number(c[6] || 0);
    const trackNum = c[7] || '';
    const filename = c[8] || '';
    const snippetUri = c[9] || '';
    const start = Number(c[10] || 0).toFixed(2);
    const end = Number(c[11] || 0).toFixed(2);
    const snippetRms = Number(c[12] || 0);
    const sourceIdx = String(c[13] ?? '');
    const mixSlug = c[14] || '';

    meta.innerHTML =
      '<b>' + title + '</b>' +
      '<br>Artists: ' + artists +
      '<br>Genre: ' + genre +
      '<br>Mix: ' + mixSlug +
      '<br>Key: ' + key +
      '<br>CSV BPM: ' + csvBpm +
      '<br>Estimated BPM: ' + fmtNum(estBpm, 2) + ' (confidence=' + fmtNum(estConf, 3) + ')' +
      '<br>Track #: ' + trackNum +
      '<br>File: ' + filename +
      '<br>Snippet: ' + start + 's → ' + end + 's | RMS: ' + fmtNum(snippetRms, 5);

    if (snippetUri) {
      audio.src = snippetUri;
      const playPromise = audio.play();
      if (playPromise && playPromise.catch) {
        playPromise.catch(() => { /* autoplay policy */ });
      }
    } else {
      audio.removeAttribute('src');
      audio.load();
    }

    const simEntry = simMap[sourceIdx] || {};
    const matches = simEntry.top_25 || [];
    const dist = simEntry.distribution || null;
    let sumProb = 0;
    for (const m of matches) sumProb += Number(m.probability || 0);

    addTop5Links(sourceIdx, matches);

    const rows = matches.map(m =>
      '<tr>' +
        '<td style="text-align:right;">' + m.rank + '</td>' +
        '<td style="text-align:right;">' + (m.track_number ?? '') + '</td>' +
        '<td>' + (m.mix_slug || '') + '</td>' +
        '<td>' + (m.title || '') + '</td>' +
        '<td>' + (m.artists || '') + '</td>' +
        '<td>' + (m.genre || '') + '</td>' +
        '<td>' + (m.key || '') + '</td>' +
        '<td style="text-align:right;">' + (m.csv_bpm || '') + '</td>' +
        '<td style="text-align:right;">' + fmtNum(m.est_bpm, 2) + '</td>' +
        '<td style="text-align:right;">' + fmtNum(m.est_conf, 3) + '</td>' +
        '<td style="text-align:right;">' + fmtNum(m.compatibility_score, 4) + '</td>' +
        '<td style="text-align:right;">' + fmtNum(m.maest_similarity, 4) + '</td>' +
        '<td style="text-align:right;">' + fmtNum(m.chroma_similarity, 4) + '</td>' +
        '<td style="text-align:right;">' + fmtNum(m.tempo_similarity, 4) + '</td>' +
        '<td style="text-align:right;">' + fmtPct(m.probability) + '</td>' +
      '</tr>'
    ).join('');

    let distHtml = '';
    if (dist) {
      distHtml =
        '<div style="margin-bottom:8px;">' +
        '<b>Multimodal probability distribution</b><br>' +
        'Top-1: <b>' + fmtPct(dist.mass_top_1) + '</b> | ' +
        'Top-5: <b>' + fmtPct(dist.mass_top_5) + '</b> | ' +
        'Top-10: <b>' + fmtPct(dist.mass_top_10) + '</b> | ' +
        'Top-25: <b>' + fmtPct(dist.mass_top_25) + '</b> | ' +
        'Remaining: <b>' + fmtPct(dist.mass_remaining) + '</b><br>' +
        <!-- 'Entropy (nats): <b>' + fmtNum(dist.entropy_nats, 4) + '</b> | ' -->
        'Active songs: <b>' + Number(dist.nonzero_count || 0) + '</b>' +
        '</div>';
    }

    let compHtml = '';
    if (matches.length > 0) {
      const m0 = matches[0];
      const wMa = Number(modelWeights.maest || 0);
      const wCh = Number(modelWeights.chroma || 0);
      const wTe = Number(modelWeights.tempo || 0);
      const cMa = wMa * Number(m0.maest_similarity || 0);
      const cCh = wCh * Number(m0.chroma_similarity || 0);
      const cTe = wTe * Number(m0.tempo_similarity || 0);
      const denom = Math.abs(cMa) + Math.abs(cCh) + Math.abs(cTe);
      const d = denom > 0 ? denom : 1.0;
      compHtml =
        '<div style="margin-bottom:8px;">' +
        '<b>Top-1 component mix</b> (' + (m0.title || '') + '):<br>' +
        'Configured weights -> MAEST: <b>' + fmtNum(wMa, 3) + '</b>, Chroma: <b>' + fmtNum(wCh, 3) + '</b>, Tempo: <b>' + fmtNum(wTe, 3) + '</b><br>' +
        'Weighted contributions -> ' +
        'MAEST: <b>' + fmtNum(cMa, 4) + '</b> (' + fmtPct(Math.abs(cMa) / d) + '), ' +
        'Chroma: <b>' + fmtNum(cCh, 4) + '</b> (' + fmtPct(Math.abs(cCh) / d) + '), ' +
        'Tempo: <b>' + fmtNum(cTe, 4) + '</b> (' + fmtPct(Math.abs(cTe) / d) + ')' +
        '</div>';
    }

    panel.innerHTML =
      '<div style="margin-bottom:6px;"><b>Top 25 Multimodal Compatible Songs</b></div>' +
      '<div style="margin-bottom:6px;">Softmax temperature: __TEMP__. ' +
      'Top-25 cumulative probability: <b>' + fmtPct(sumProb) + '</b></div>' +
      compHtml +
      distHtml +
      '<div style="max-height:360px;overflow:auto;border:1px solid #eee;">' +
      '<table style="width:100%;border-collapse:collapse;font-size:12px;">' +
      '<thead><tr>' +
      '<th style="text-align:right;">#</th>' +
      '<th style="text-align:right;">Track</th>' +
      '<th>Mix</th>' +
      '<th>Title</th>' +
      '<th>Artists</th>' +
      '<th>Genre</th>' +
      '<th>Key</th>' +
      '<th style="text-align:right;">CSV BPM</th>' +
      '<th style="text-align:right;">Est BPM</th>' +
      '<th style="text-align:right;">Conf</th>' +
      '<th style="text-align:right;">Score</th>' +
      '<th style="text-align:right;">MAEST</th>' +
      '<th style="text-align:right;">Chroma</th>' +
      '<th style="text-align:right;">Tempo</th>' +
      '<th style="text-align:right;">Prob</th>' +
      '</tr></thead><tbody>' + rows + '</tbody></table></div>';
  });
})();
</script>
'''
script = (
    script
    .replace('__SIM_MAP__', sim_json)
    .replace('__PLOT_ID__', plot_div_id)
    .replace('__TEMP__', str(MULTIMODAL_TEMPERATURE))
    .replace('__MODEL_WEIGHTS__', weights_json)
    .replace('__IDX_TO_POINT__', idx_to_point_json)
)

interactive_html = '\n'.join([
    '<!doctype html>',
    '<html><head><meta charset="utf-8"></head><body style="font-family:Arial,sans-serif;">',
    plot_html,
    meta_audio_html,
    script,
    '</body></html>',
])

INTERACTIVE_HTML_PATH.parent.mkdir(parents=True, exist_ok=True)
INTERACTIVE_HTML_PATH.write_text(interactive_html, encoding='utf-8')

missing_snippets = sum(1 for r in records if not r['snippet_uri'])
print(f'Loaded aligned tracks: {len(records)}')
print(f'Prepared snippets: {len(records) - missing_snippets}/{len(records)} tracks')
print('Snippet cache dirs:')
for cfg in MIX_CONFIGS:
    print(f"  - {Path(cfg['snippet_cache_dir'])}")
print('Similarity: multimodal compatibility (MAEST+Chroma+Tempo) with explicit harmonic + tempo-sim params.')
print(f'Standalone HTML saved to: {INTERACTIVE_HTML_PATH}')
print('To pre-generate snippets explicitly, run:')
for cfg in MIX_CONFIGS:
    print(f"  uv run djprojectexploration-snippet-cache {Path(cfg['tracklist_csv'])} --music-dir {Path(cfg['music_dir'])}")

try:
    opened = webbrowser.open_new_tab(INTERACTIVE_HTML_PATH.resolve().as_uri())
    print(f'Opened in browser: {opened}')
except Exception as exc:
    print(f'Could not auto-open browser ({type(exc).__name__}): {exc}')


Loaded aligned tracks: 48
Prepared snippets: 48/48 tracks
Snippet cache dirs:
  - /Users/josephdaher/Git Repositories/djprojectexploration/data/snippets/aries_mix_tracks
  - /Users/josephdaher/Git Repositories/djprojectexploration/data/snippets/ara_mix_tracks
Similarity: multimodal compatibility (MAEST+Chroma+Tempo) with explicit harmonic + tempo-sim params.
Standalone HTML saved to: /Users/josephdaher/Git Repositories/djprojectexploration/data/exports/aries_mix_tracks__ara_mix_tracks_interactive_pacmap_multimodal.html
To pre-generate snippets explicitly, run:
  uv run djprojectexploration-snippet-cache /Users/josephdaher/Git Repositories/djprojectexploration/music/aries-mix/aries_mix_tracks.csv --music-dir /Users/josephdaher/Git Repositories/djprojectexploration/music/aries-mix
  uv run djprojectexploration-snippet-cache /Users/josephdaher/Git Repositories/djprojectexploration/music/ara-mix/ara_mix_tracks.csv --music-dir /Users/josephdaher/Git Repositories/djprojectexploration/music/a